# Time Zone Handling

Working with time zones can be one of the most unpleasant parts of time series manipulation. As a result, many users choose to work with time series in **coordinated universal time**, or **UTC** — the geography-independent international standard. Time zones are expressed as offsets from UTC; for example, New York is four hours behind UTC during daylight saving time (DST) and five hours behind the rest of the year.

Time zone data itself (which regions observe DST, when the offsets change, historical rule changes, etc.) comes from the **IANA Time Zone Database** — often still called the *Olson database* after its original maintainer. In Python this data is exposed in two places:

- **`pytz`** — a third-party library, historically the way pandas did all of its time zone work.
- **`zoneinfo`** (the standard library, Python 3.9+) — what modern pandas actually reaches for by default now.

> **Nuance:** older material (including earlier editions of the book this notebook is based on) says pandas has a *hard dependency* on `pytz`. That's no longer true — `pytz` is now an *optional* extra (`pip install pandas[timezone]`). You can pass either a `pytz` timezone object or a plain string to pandas' time zone methods; when you pass a string, pandas resolves it via `zoneinfo` under the hood, not `pytz`, even if `pytz` happens to be installed. This notebook still shows `pytz.timezone(...)` because it's a useful way to browse available zone names, but for everyday use you'll almost always just pass strings.

Time zone names can be found interactively and in the docs.

In [1]:
import pytz 
import pandas as pd 
import numpy as np

pytz.common_timezones[-5:]

['US/Eastern', 'US/Hawaii', 'US/Mountain', 'US/Pacific', 'UTC']

To get a time zone object from `pytz`, use `pytz.timezone`:

In [2]:
tz = pytz.timezone("America/New_York")

tz

<DstTzInfo 'America/New_York' LMT-1 day, 19:04:00 STD>

Here's the "pandas actually uses `zoneinfo`, not `pytz`" claim made concrete — pass a string to `pandas.Timestamp` and check what kind of object comes back out:

In [3]:
type(pd.Timestamp("2011-03-12 04:00", tz="America/New_York").tzinfo)

zoneinfo.ZoneInfo

Methods in pandas will accept either time zone names (strings) or `pytz`/`zoneinfo` objects directly.

## Time Zone Localization and Conversion

> **Definitions worth pinning down before going further:**
> - **Time zone *naive***: a timestamp with no time zone attached at all. `2012-03-09 09:30:00` on its own is ambiguous — 9:30am *where*?
> - **Time zone *aware***: a timestamp with a specific time zone (or UTC offset) attached, so it unambiguously identifies one exact instant.
>
> By default, time series in pandas are naive. For example, consider the following time series:

In [4]:
dates = pd.date_range("2012-03-09 09:30", periods=6)

ts = pd.Series(np.random.standard_normal(len(dates)), index=dates)

ts

2012-03-09 09:30:00    0.315403
2012-03-10 09:30:00   -1.114739
2012-03-11 09:30:00    1.318375
2012-03-12 09:30:00   -1.124287
2012-03-13 09:30:00   -1.955800
2012-03-14 09:30:00   -1.970789
Freq: D, dtype: float64

The index's `tz` field is `None`:

In [5]:
print(ts.index.tz)

None


Date ranges can be generated with a time zone set:

In [6]:
pd.date_range("2012-03-09 09:30", periods=10, tz="UTC")

DatetimeIndex(['2012-03-09 09:30:00+00:00', '2012-03-10 09:30:00+00:00',
               '2012-03-11 09:30:00+00:00', '2012-03-12 09:30:00+00:00',
               '2012-03-13 09:30:00+00:00', '2012-03-14 09:30:00+00:00',
               '2012-03-15 09:30:00+00:00', '2012-03-16 09:30:00+00:00',
               '2012-03-17 09:30:00+00:00', '2012-03-18 09:30:00+00:00'],
              dtype='datetime64[us, UTC]', freq='D')

Conversion from naive to *localized* (reinterpreted as having been observed in a particular time zone) is handled by the `tz_localize` method.

> **The distinction that trips people up:** `tz_localize` **does not change any clock values** — it just attaches a time zone label to numbers that are already there, on the assumption that they were always meant to represent that zone. `tz_convert`, covered next, is the opposite: it **does** change the clock values, shifting them to whatever the same real instant looks like in a different zone. Watch the clock times below stay identical (`09:30`) after `tz_localize` — only a UTC offset gets added:

In [7]:
ts

2012-03-09 09:30:00    0.315403
2012-03-10 09:30:00   -1.114739
2012-03-11 09:30:00    1.318375
2012-03-12 09:30:00   -1.124287
2012-03-13 09:30:00   -1.955800
2012-03-14 09:30:00   -1.970789
Freq: D, dtype: float64

In [8]:
ts_utc = ts.tz_localize("UTC")

ts_utc

2012-03-09 09:30:00+00:00    0.315403
2012-03-10 09:30:00+00:00   -1.114739
2012-03-11 09:30:00+00:00    1.318375
2012-03-12 09:30:00+00:00   -1.124287
2012-03-13 09:30:00+00:00   -1.955800
2012-03-14 09:30:00+00:00   -1.970789
Freq: D, dtype: float64

In [9]:
ts_utc.index

DatetimeIndex(['2012-03-09 09:30:00+00:00', '2012-03-10 09:30:00+00:00',
               '2012-03-11 09:30:00+00:00', '2012-03-12 09:30:00+00:00',
               '2012-03-13 09:30:00+00:00', '2012-03-14 09:30:00+00:00'],
              dtype='datetime64[us, UTC]', freq='D')

Once a time series has been localized to a particular time zone, it can be converted to another time zone with `tz_convert`. Unlike `tz_localize`, this **does** change the wall-clock time — it's answering "what time was it in New York, at the exact same instant?", not just relabeling. Compare the clock values here (`04:30`/`05:30`) to the identical instant's `09:30` UTC above:

In [10]:
ts_utc.tz_convert("America/New_York")

2012-03-09 04:30:00-05:00    0.315403
2012-03-10 04:30:00-05:00   -1.114739
2012-03-11 05:30:00-04:00    1.318375
2012-03-12 05:30:00-04:00   -1.124287
2012-03-13 05:30:00-04:00   -1.955800
2012-03-14 05:30:00-04:00   -1.970789
dtype: float64

In the case of the preceding time series, which straddles a DST transition in the `America/New_York` time zone, we could localize to US Eastern time and convert to, say, UTC or Berlin time — notice the Eastern UTC offset changes partway through (`-05:00` → `-04:00`) as the series crosses the US DST boundary on March 11. The EU's DST transition falls later in March (the last Sunday, not the second), so the Berlin conversion stays at a constant `+01:00` for this same date range — a good reminder that different regions change clocks on different dates:

In [11]:
ts_eastern = ts.tz_localize("America/New_York")

ts_eastern.tz_convert("UTC")

2012-03-09 14:30:00+00:00    0.315403
2012-03-10 14:30:00+00:00   -1.114739
2012-03-11 13:30:00+00:00    1.318375
2012-03-12 13:30:00+00:00   -1.124287
2012-03-13 13:30:00+00:00   -1.955800
2012-03-14 13:30:00+00:00   -1.970789
dtype: float64

In [12]:
ts_eastern.tz_convert("Europe/Berlin")

2012-03-09 15:30:00+01:00    0.315403
2012-03-10 15:30:00+01:00   -1.114739
2012-03-11 14:30:00+01:00    1.318375
2012-03-12 14:30:00+01:00   -1.124287
2012-03-13 14:30:00+01:00   -1.955800
2012-03-14 14:30:00+01:00   -1.970789
dtype: float64

`tz_localize` and `tz_convert` are also instance methods on `DatetimeIndex` directly, not just on a Series:

In [13]:
ts.index.tz_localize("Asia/Shanghai")

DatetimeIndex(['2012-03-09 09:30:00+08:00', '2012-03-10 09:30:00+08:00',
               '2012-03-11 09:30:00+08:00', '2012-03-12 09:30:00+08:00',
               '2012-03-13 09:30:00+08:00', '2012-03-14 09:30:00+08:00'],
              dtype='datetime64[us, Asia/Shanghai]', freq=None)

## Operations with Time Zone-Aware Timestamp Objects

Similar to time series and date ranges, individual `Timestamp` objects similarly can be localized from naive to time zone-aware and converted from one time zone to another:

In [14]:
stamp = pd.Timestamp("2011-03-12 04:00")

stamp_utc = stamp.tz_localize("utc")

stamp_utc.tz_convert("America/New_York")

Timestamp('2011-03-11 23:00:00-0500', tz='America/New_York')

You can also pass a time zone when creating the `Timestamp`:

In [15]:
stamp_moscow = pd.Timestamp("2011-03-12 04:00", tz="Europe/Moscow")

stamp_moscow

Timestamp('2011-03-12 04:00:00+0300', tz='Europe/Moscow')

Time zone-aware `Timestamp` objects internally store a single UTC instant, so changing the time zone only changes how it's *displayed* — the underlying value doesn't move. `.value` exposes that underlying number as nanoseconds since the Unix epoch (January 1, 1970 UTC) — note this is always nanoseconds for backward compatibility, even though (per the previous notebook's resolution-flexibility nuance) the `Timestamp` itself might actually be stored at microsecond resolution internally:

In [16]:
stamp_utc.value

1299902400000000000

In [17]:
stamp_utc.tz_convert("America/New_York").value

1299902400000000000

## Arithmetic Across DST Transitions

Two quick definitions, since the US switches on different dates than much of the world and the terminology matters here:
- **Spring forward**: clocks jump *ahead* one hour (2:00am → 3:00am) when DST *begins*. The hour from 2:00–2:59am **doesn't exist** that day.
- **Fall back**: clocks move *back* one hour (2:00am → 1:00am) when DST *ends*. The hour from 1:00–1:59am **happens twice** that day.

When you add a `DateOffset` like `Hour()` to a time zone-aware `Timestamp`, pandas doesn't naively add "1" to the hour field on the clock face. It converts to the underlying UTC instant, adds the real elapsed duration there, and *then* converts the result back to local wall-clock time. That's what "respects DST transitions" means in practice — the wall-clock hour you see can jump by more or less than the number you asked for, because real elapsed time and wall-clock-displayed time briefly disagree during a transition.

First, a timestamp 30 minutes before the spring-forward transition:

In [18]:
stamp = pd.Timestamp("2012-03-11 01:30", tz="US/Eastern")

stamp

Timestamp('2012-03-11 01:30:00-0500', tz='US/Eastern')

In [19]:
from pandas.tseries.offsets import Hour

stamp + Hour()

Timestamp('2012-03-11 03:30:00-0400', tz='US/Eastern')

`01:30 EST` is `06:30 UTC`. Adding one real hour lands on `07:30 UTC` — which is already past the `07:00 UTC` transition moment (2:00am EST becomes 3:00am EDT at that instant), so it displays as `03:30 EDT`, not the nonexistent `02:30`. One real hour passed; the wall clock jumped by two.

Then, the fall-back case — 90 minutes before transitioning out of DST:

In [20]:
stamp = pd.Timestamp("2012-11-04 00:30", tz="US/Eastern")

stamp

Timestamp('2012-11-04 00:30:00-0400', tz='US/Eastern')

In [21]:
stamp + 2 * Hour()

Timestamp('2012-11-04 01:30:00-0500', tz='US/Eastern')

`00:30 EDT` is `04:30 UTC`; adding two real hours lands on `06:30 UTC`, which is past the `06:00 UTC` transition (2:00am EDT becomes 1:00am EST at that instant), so it displays as `01:30 EST`. Two real hours passed, but the wall clock only advanced by one — the mirror image of the spring-forward case. This is exactly why naive "just add to the hour field" arithmetic breaks around DST: the mapping between wall-clock time and real elapsed time isn't one-to-one on transition days.

**Gotcha:** this real-elapsed-time behavior only applies to *arithmetic* on an already-aware `Timestamp`. If you instead try to directly `tz_localize` a naive wall-clock time that falls in the skipped or repeated hour, pandas can't guess what you meant and raises a `ValueError`:

In [22]:
try:
    pd.Timestamp("2012-03-11 02:30").tz_localize("US/Eastern")  # 2:30am doesn't exist on this day
except ValueError as e:
    print(e)

2012-03-11 02:30:00 is a nonexistent time due to daylight savings time. Try using the 'nonexistent' argument.


The fall-back side has the opposite problem — `01:30` occurs *twice*, so pandas can't tell which one you mean:

In [23]:
try:
    pd.Timestamp("2012-11-04 01:30").tz_localize("US/Eastern")  # 1:30am happens twice on this day
except ValueError as e:
    print(e)

Cannot infer dst time from 2012-11-04 01:30:00, try using the 'ambiguous' argument


Both error messages point at the fix: pass `nonexistent="shift_forward"` (or `"NaT"`) to push a nonexistent time past the gap, or `ambiguous="NaT"` / `ambiguous=True`/`False` to pick a side of a repeated hour, when you actually have data landing in one of these edge cases.

## Operations Between Different Time Zones

If two time series with different time zones are combined, the result will be UTC. Since aware timestamps are stored under the hood as UTC instants (as established above), this is a straightforward operation and requires no explicit conversion — pandas just aligns on the shared UTC representation and the zone labels along the way become irrelevant:

In [24]:
dates = pd.date_range("2012-03-07 09:30", periods=10, freq="B")

ts = pd.Series(np.random.standard_normal(len(dates)), index=dates)

ts

2012-03-07 09:30:00   -0.585155
2012-03-08 09:30:00   -0.048583
2012-03-09 09:30:00    0.204206
2012-03-12 09:30:00    0.454866
2012-03-13 09:30:00    0.006251
2012-03-14 09:30:00   -1.224383
2012-03-15 09:30:00    0.468462
2012-03-16 09:30:00    0.186562
2012-03-19 09:30:00    1.930262
2012-03-20 09:30:00   -0.194152
Freq: B, dtype: float64

In [25]:
ts1 = ts[:7].tz_localize("Europe/London")

ts2 = ts1[2:].tz_convert("Europe/Moscow")

result = ts1 + ts2

result.index

DatetimeIndex(['2012-03-07 09:30:00+00:00', '2012-03-08 09:30:00+00:00',
               '2012-03-09 09:30:00+00:00', '2012-03-12 09:30:00+00:00',
               '2012-03-13 09:30:00+00:00', '2012-03-14 09:30:00+00:00',
               '2012-03-15 09:30:00+00:00'],
              dtype='datetime64[us, UTC]', freq=None)

Operations between a time zone-naive and a time zone-aware series are not supported and will raise an exception — pandas has no way to know what zone the naive side's clock times were meant to represent, so it refuses to guess:

In [26]:
try:
    ts + ts1  # ts is naive, ts1 is tz-aware (Europe/London)
except TypeError as e:
    print(e)

Cannot join tz-naive with tz-aware DatetimeIndex


---

## Summary / Cheat Sheet

**The two core operations, and how they differ:**

| | `tz_localize` | `tz_convert` |
|---|---|---|
| Input | Naive timestamp | Already tz-aware timestamp |
| What changes | Nothing — just attaches a zone label | The displayed wall-clock time |
| What stays the same | The clock numbers you see | The real underlying UTC instant |
| Mental model | "This `09:30` was always New York time" | "What does that same instant look like in Berlin?" |

**Nuances worth remembering:**
- Time zone-aware objects are stored internally as a single UTC instant; the zone is just a display lens. That's why combining series across zones "just works" (everything's UTC underneath) and why combining naive with aware doesn't (there's no instant to align on).
- Modern pandas resolves string zone names (`"America/New_York"`) via the standard library's `zoneinfo`, not `pytz` — `pytz` is now optional, despite what older material says.
- `DateOffset` arithmetic on an aware `Timestamp` operates on **real elapsed time**, not the wall-clock hour field — which is exactly what lets it "respect DST": the displayed hour can jump by more or less than you asked for right at a transition.
- Directly `tz_localize`-ing a naive time that falls in the spring-forward gap or the fall-back repeat raises `ValueError` — pandas won't guess; use `nonexistent=` / `ambiguous=` to say what you mean.
- `.value` is always nanoseconds, regardless of the `Timestamp`'s actual storage resolution (see the previous notebook's note on flexible resolution) — it's kept at nanosecond scale for backward compatibility.

**Up next:** *Periods and Period Arithmetic* — representing a *span* of time (like "all of March 2012") rather than a single instant, and converting between `Period` and `Timestamp`.